# Natural Language Processing

## Предварительная обработка текстов

### Задача классификации твитов на тональность

In [30]:
# скачаем куски датасета
!wget https://raw.githubusercontent.com/maryszmary/netology_nlp_2021/master/sem1/tweets_sentiment.csv

--2025-09-13 17:26:45--  https://raw.githubusercontent.com/maryszmary/netology_nlp_2021/master/sem1/tweets_sentiment.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32795904 (31M) [text/plain]
Saving to: ‘tweets_sentiment.csv’

tweets_sentiment.cs 100%[===================>]  31.28M  22.9MB/s    in 1.4s    

2025-09-13 17:26:48 (22.9 MB/s) - ‘tweets_sentiment.csv’ saved [32795904/32795904]



In [74]:
from string import punctuation
from collections import Counter

import pandas as pd
import numpy as np
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from nltk import ngrams
from nltk import tokenize
from nltk.tokenize import word_tokenize, toktok

In [32]:
df = pd.read_csv('tweets_sentiment.csv')

In [33]:
df.head()

,text,label
0,мыс на меня обиделась:(\nя ей даже ничего не с...,negative
1,"аааааааааааааааааааа,не хочу на работу :(",negative
2,"У меня какой-то особенный вид ушей! :D, некото...",positive
3,@simonovkon он неплохой человек в жизни. Я ра...,negative
4,"RT @Darina_Lo: Домааааа\nЕхали на такси, пели ...",positive


In [34]:
x_train, x_test, y_train, y_test = train_test_split(df.text, df.label)

In [35]:
print(df.shape)
print(x_train.shape)
print(x_test.shape)

(226834, 2)
(170125,)
(56709,)


### Baseline: Классификация необработанных n-грамм
### Векторизаторы

In [36]:
df['text'].head().tolist()

['мыс на меня обиделась:(\nя ей даже ничего не сделала:(',
 'аааааааааааааааааааа,не хочу на работу :(',
 'У меня какой-то особенный вид ушей! :D, некоторые вакуумные наушники в моих ушах просто не держатся!',
 '@simonovkon  он неплохой человек в жизни. Я работала в шоу-бизе, со многими знакома. Встречаются очень хорошие люди. И не очень(((',
 'RT @Darina_Lo: Домааааа\nЕхали на такси, пели песни, отдыхали.\nКричали на улице:)\nМы настоящяя семья)']

#### CountVectorizer

- Строит для каждого документа (каждой пришедшей ему строки) вектор размерности `n`, где `n` - количество слов или n-грамм во всем корпусе
- Заполняет каждый i-й элемент количеством вхождений слова в данный документ

In [37]:
vec = CountVectorizer(ngram_range=(1, 1))
bow = vec.fit_transform(x_train)

In [38]:
list(vec.vocabulary_.items())[:20]

[('rt', 74752),
 ('artem_klyushin', 14722),
 ('устал', 228985),
 ('как', 142979),
 ('собака', 212769),
 ('выходные', 118515),
 ('хочу', 234620),
 ('провести', 195778),
 ('дома', 127712),
 ('играя', 139422),
 ('онлайн', 173877),
 ('игры', 139473),
 ('jemapelka', 42301),
 ('your_novocaine', 95247),
 ('мы', 161605),
 ('тобой', 222436),
 ('только', 222725),
 ('вчера', 116461),
 ('переписывались', 181211),
 ('shadwil', 77627)]

#### Создание n-грамм с помощью библиотеки nltk

In [40]:
sent = 'Harry Potter and the Methods of Rationality'.split()
list(ngrams(sent,1)) # униграммы

[('Harry',),
 ('Potter',),
 ('and',),
 ('the',),
 ('Methods',),
 ('of',),
 ('Rationality',)]

In [41]:
sent

['Harry', 'Potter', 'and', 'the', 'Methods', 'of', 'Rationality']

In [42]:
list(ngrams(sent, 2)) # биграммы

[('Harry', 'Potter'),
 ('Potter', 'and'),
 ('and', 'the'),
 ('the', 'Methods'),
 ('Methods', 'of'),
 ('of', 'Rationality')]

In [43]:
list(ngrams(sent, 3)) # триграммы

[('Harry', 'Potter', 'and'),
 ('Potter', 'and', 'the'),
 ('and', 'the', 'Methods'),
 ('the', 'Methods', 'of'),
 ('Methods', 'of', 'Rationality')]

In [ ]:
# Обучение модели логистической регрессии:
clf = LogisticRegression(random_state=42, solver='liblinear')
clf.fit(bow, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [45]:
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.77      0.76      0.76     28234
    positive       0.76      0.77      0.77     28475

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



In [47]:
# Обучение модели на триграммах:
vec = CountVectorizer(ngram_range=(3, 3))
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state=42)
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.47      0.72      0.57     18366
    positive       0.82      0.61      0.70     38343

    accuracy                           0.65     56709
   macro avg       0.65      0.67      0.64     56709
weighted avg       0.71      0.65      0.66     56709



#### Эксперименты с CountVectorizer

In [48]:
vec_mini = CountVectorizer()

In [49]:
corpus = [
    'в москве сегодня снег',
    'в москве холодно и сыро',
    'снег это хорошо но холодно',
]

In [50]:
vec_mini.fit(corpus)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [51]:
vec_mini.vocabulary_

{'москве': 0,
 'сегодня': 2,
 'снег': 3,
 'холодно': 5,
 'сыро': 4,
 'это': 7,
 'хорошо': 6,
 'но': 1}

In [53]:
matrix = vec_mini.transform(corpus)
matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 11 stored elements and shape (3, 8)>

In [54]:
matrix.todense()

matrix([[1, 0, 1, 1, 0, 0, 0, 0],
        [1, 0, 0, 0, 1, 1, 0, 0],
        [0, 1, 0, 1, 0, 1, 1, 1]])

In [55]:
matrix = vec_mini.transform(['в москве живут котики'])
matrix.todense()

matrix([[1, 0, 0, 0, 0, 0, 0, 0]])

#### Tfidf Vectorizer

In [57]:
vec = TfidfVectorizer()
bow = vec.fit_transform(x_train)
clf = LogisticRegression(random_state=42, solver='liblinear')
clf.fit(bow, y_train)
pred = clf.predict(vec.transform(x_test))
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

    negative       0.73      0.77      0.75     26772
    positive       0.78      0.75      0.77     29937

    accuracy                           0.76     56709
   macro avg       0.76      0.76      0.76     56709
weighted avg       0.76      0.76      0.76     56709



### Токенизация

In [58]:
# split по пробелу
print(df['text'].iloc[0].split())

['мыс', 'на', 'меня', 'обиделась:(', 'я', 'ей', 'даже', 'ничего', 'не', 'сделала:(']


In [60]:
vec.get_feature_names_out()[20:40]

array(['003', '003r38hn6e', '004anna', '004hafarf4', '005', '0060', '007',
       '0080', '008ge0nygh', '009', '00_elenka', '00_katusha',
       '00_orekhova', '00darya', '00ennqulcp', '00fedosova',
       '00gorbunova', '00kudrina', '00lg6bsnb8', '00ngi3vmps'],
      dtype=object)

#### Использование nltk.rokenize

In [63]:
example = 'Но не каждый хочет что-то исправлять:('
word_tokenize(example)

['Но', 'не', 'каждый', 'хочет', 'что-то', 'исправлять', ':', '(']

In [64]:
text = u'Is 9.5 or 525,600 my favorite number?'
word_tokenize(text)

['Is', '9.5', 'or', '525,600', 'my', 'favorite', 'number', '?']

In [67]:
dir(tokenize)[:16]

['BlanklineTokenizer',
 'LegalitySyllableTokenizer',
 'LineTokenizer',
 'MWETokenizer',
 'NLTKWordTokenizer',
 'PunktSentenceTokenizer',
 'PunktTokenizer',
 'RegexpTokenizer',
 'ReppTokenizer',
 'SExprTokenizer',
 'SpaceTokenizer',
 'StanfordSegmenter',
 'SyllableTokenizer',
 'TabTokenizer',
 'TextTilingTokenizer',
 'ToktokTokenizer']

In [68]:
toktok = toktok.ToktokTokenizer()
text = u'Is 9.5 or 525,600 my favorite number?'
print(toktok.tokenize(text))

['Is', '9.5', 'or', '525,600', 'my', 'favorite', 'number', '?']


Получение индексов начала и конца каждого токена

In [69]:
wh_tok = tokenize.WhitespaceTokenizer()
list(wh_tok.span_tokenize(example))

[(0, 2), (3, 5), (6, 12), (13, 18), (19, 25), (26, 38)]

### Самые частотные слова

In [76]:
# стандартная библиотека string
punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [77]:
# стандатрная библиотека collections
corpus = [token for tweet in df.text for token in word_tokenize(tweet) if token not in punctuation]
print(len(corpus))
corpus[:10]

2870536


['мыс',
 'на',
 'меня',
 'обиделась',
 'я',
 'ей',
 'даже',
 'ничего',
 'не',
 'сделала']

In [78]:
# частотный словарь
freq_dict = Counter(corpus)

freq_dict.most_common(20)

[('не', 69472),
 ('и', 55166),
 ('в', 52902),
 ('я', 52818),
 ('RT', 38070),
 ('на', 35759),
 ('http', 32998),
 ('что', 31541),
 ('с', 27217),
 ('а', 26860),
 ('...', 22363),
 ('меня', 20656),
 ('у', 18928),
 ('как', 18280),
 ('так', 16839),
 ('D', 16575),
 ('это', 16542),
 ('мне', 16337),
 ('все', 14763),
 ('ты', 13412)]

### Стоп-слова и пунктуация